# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hammadkhaliq-del/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

**What this notebook does, stated up front:** two parts. First, the same methodology audit applied to two findings in FlyRank's own March 2026 research paper (`docs/flyrank-seo-research-march-2026.pdf`) that we walked in the live session. Second, that same lens turned on my own Week-5 model (`w05_model.ipynb`): a before/after honest-split comparison, a leakage audit against the checklist in `hunting-leakage-and-validating`, and a rewrite of my boldest w05 claim into safe language.

All claims below use: observed, measured, directional, decision-support — not "proves," "guarantees," or "causes."


In [1]:
%pip install -q duckdb scikit-learn

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('AccessToken')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FEATURE_MONTH = "2026-03"
LABEL_MONTH   = "2026-04"

DAILY_FEATURE = f"{REL}/fact_content_daily_performance/month={FEATURE_MONTH}/data_0.parquet"
DAILY_LABEL   = f"{REL}/fact_content_daily_performance/month={LABEL_MONTH}/data_0.parquet"

probe = con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{DAILY_LABEL}')").df()
print(f'{LABEL_MONTH} partition row count:', probe['n'].iloc[0])
assert probe['n'].iloc[0] > 0, f'{LABEL_MONTH} has no rows under this path — fix the path before continuing.'
print('Ready.')


2026-04 partition row count: 10424730
Ready.


## 1. Two paper findings + my methodology questions

I'm picking two findings the paper itself flags as needing care — not because they're weak, but because the paper's own methodology page invites exactly this kind of question ("ML pages are exploratory... importance is descriptive rather than causal"). Constructive tone throughout: this is the same rigor I'm about to apply to my own notebook, not a takedown.

### Finding #4 — "The Freshness Multiplier" (365+ day refresh → 3.2x health boost)

**The claim:** 365+ day content refreshed within 30 days shows a 3.2x health-score boost (10.7 → 34.5) and 57x more impressions (71 → 4,039), presented as "one of the strongest measured levers available."

**Where does the label come from?** The paper doesn't fully specify the population the 3.2x is computed over. Is it a true before/after on the *same pages* (page P at health 10.7, then measured again post-refresh at 34.5), or a cross-sectional comparison between a "365+ refreshed" group and a "365+ not refreshed" group at a single point in time? Those are very different claims. A true before/after on the same pages supports "refreshing this specific page lifted its score." A cross-sectional comparison only supports "pages that got refreshed tend to score higher" — which opens the door to selection: content teams likely chose to refresh pages that already had *some* residual signal (existing backlinks, a topic still in demand) worth saving, rather than refreshing at random. The paper's own limitations page names "content age confounds model-performance comparisons" — but doesn't say whether refresh *selection* itself is confounded the same way.

**Does the validation design support the claim?** The paper is explicit and honest that the `361+` bucket is "too small and too unstable to treat as a headline multiplier" (283:1 growth ratio on just 1 declining page) — that transparency is good practice and worth naming. But the 3.2x/57x figures sit right next to that same unstable bucket, and the paper doesn't state the *n* behind the 3.2x/57x pair specifically (is it also a handful of pages?). A methodology question worth asking: what is n for the refreshed-365+ group, and is the 57x impression figure driven by a small number of high-impression outliers (the paper elsewhere shows this pattern — e.g. the 212k-impression / near-zero-CTR row flagged in my own Week-4 baseline review)? Without n and a spread (not just two point estimates), a reader can't judge how repeatable the multiplier is.

### ML Appendix — "What Predicts Health?" (Random Forest, Average Position 43% importance)

**The claim:** A Random Forest predicting Health Score finds Average Position the top feature (43% importance), followed by Impressions (32%) and Scroll Depth (15%).

**Where does the label come from?** The paper discloses this directly and I want to credit that: "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal." Health Score is stated on the methodology page as `Impressions (30pts) + Position (30pts) + CTR (20pts) + Scroll Depth (20pts)` — a weighted sum of three of the model's own top features. This is the exact label-derived-feature pattern in the leakage skill's taxonomy: "the label was computed FROM a column, and that column is in the features... symptom: one feature towers over all others." The paper's self-aware framing is the right instinct, but it stops one step short of the taxonomy's own test: *train once WITH Position/Impressions/Scroll Depth, once WITHOUT, and show the importance-ranking collapse.* Without that ablation, a reader can't tell how much of the 43%/32%/15% is "the model found something real about what makes content healthy" versus "the model found the arithmetic identity used to build its own target."

**Does the validation design support the claim?** The methodology page states an 80/20 split for the Random Forest, with no mention of grouping by client or brand. With 57 brands and 61.8K content pieces, an ungrouped random split likely puts many pages from the same brand in both train and test — meaning some of that 43%/32%/15% importance could reflect the model learning brand-level baseline patterns (a brand's typical position/impression scale) rather than a generalizable content-health relationship. This mirrors exactly the split risk in my own Week-5 work before I grouped by `client_hash_id`.

## 2. My model under an honest split (before/after)

*This is the part of the leakage skill's advice I'm applying most directly: "Swap your random split for a grouped split and report both numbers. If you can't explain the gap, you're not done."*

My Week-5 notebook already used `GroupShuffleSplit` by `client_hash_id` — the "after." To make this an honest before/after rather than restating a decision I'd already made, I'm rebuilding the "before": the same model, same features, same label, but a naive random row split with no grouping. That's the split a first pass at this problem would plausibly use, and it's the split the paper itself appears to use for its Random Forest and Logistic Regression appendices.


In [2]:
# --- Rebuild the March (feature) / April (label) frame, same construction as w05_model.ipynb ---
feature_month_df = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS impressions_month,
           SUM(gsc_clicks) AS clicks_month,
           AVG(gsc_avg_position) AS avg_position_month
    FROM read_parquet('{DAILY_FEATURE}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()
feature_month_df['ctr_month'] = feature_month_df['clicks_month'] / feature_month_df['impressions_month'] * 100

def position_bucket(pos):
    if pos <= 0: return '0_no_position_data'
    elif pos <= 3: return '1_top3'
    elif pos <= 10: return '2_page1_4_10'
    elif pos <= 20: return '3_page2_11_20'
    elif pos <= 50: return '4_page3plus_21_50'
    else: return '5_deep_50plus'

feature_month_df['position_bucket'] = feature_month_df['avg_position_month'].apply(position_bucket)

label_month_df = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS impressions_next
    FROM read_parquet('{DAILY_LABEL}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

data = feature_month_df.merge(label_month_df, on=['content_hash_id', 'client_hash_id'], how='inner')
data['pct_change'] = (data['impressions_next'] - data['impressions_month']) / data['impressions_month'] * 100
data['is_declining_next'] = (data['pct_change'] < -20).astype(int)
data['log_impressions_month'] = np.log1p(data['impressions_month'])
data['log_clicks_month'] = np.log1p(data['clicks_month'])

print('Rows:', len(data), '| Distinct clients:', data['client_hash_id'].nunique())
print('Base rate (is_declining_next=1):', round(data['is_declining_next'].mean(), 4))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 158549 | Distinct clients: 46
Base rate (is_declining_next=1): 0.4782


In [3]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

num_features = ['impressions_month', 'clicks_month', 'avg_position_month', 'ctr_month',
                'log_impressions_month', 'log_clicks_month']
cat_features = ['position_bucket']

pre = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
])

def fit_eval(train_df, test_df, tag):
    X_train, y_train = train_df[num_features + cat_features], train_df['is_declining_next']
    X_test,  y_test  = test_df[num_features + cat_features],  test_df['is_declining_next']
    pipe = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = pipe.predict(X_test)
    base_rate = y_test.mean()
    return {
        'split': tag,
        'n_train': len(train_df), 'n_test': len(test_df),
        'test_base_rate': round(base_rate, 3),
        'roc_auc': round(roc_auc_score(y_test, proba), 3),
        'precision': round(precision_score(y_test, pred, zero_division=0), 3),
        'recall': round(recall_score(y_test, pred, zero_division=0), 3),
        'f1': round(f1_score(y_test, pred, zero_division=0), 3),
    }

# --- BEFORE: naive random row split, no grouping ---
train_rand, test_rand = train_test_split(data, test_size=0.25, random_state=42, stratify=data['is_declining_next'])
rand_overlap_clients = set(train_rand['client_hash_id']) & set(test_rand['client_hash_id'])
result_random = fit_eval(train_rand, test_rand, 'BEFORE: random row split')
result_random['clients_leaked_across_split'] = len(rand_overlap_clients)

# --- AFTER: grouped by client, same as w05 ---
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(data, groups=data['client_hash_id']))
train_grp = data.iloc[train_idx].reset_index(drop=True)
test_grp  = data.iloc[test_idx].reset_index(drop=True)
grp_overlap_clients = set(train_grp['client_hash_id']) & set(test_grp['client_hash_id'])
assert len(grp_overlap_clients) == 0, 'Client leakage in the grouped split — should be impossible by construction'
result_grouped = fit_eval(train_grp, test_grp, 'AFTER: grouped by client')
result_grouped['clients_leaked_across_split'] = len(grp_overlap_clients)

before_after = pd.DataFrame([result_random, result_grouped]).set_index('split')
before_after


,n_train,n_test,test_base_rate,roc_auc,precision,recall,f1,clients_leaked_across_split
split,,,,,,,,
BEFORE: random row split,118911,39638,0.478,0.628,0.568,0.595,0.581,45
AFTER: grouped by client,135314,23235,0.470,0.664,0.586,0.647,0.615,0


*(Fill in after running — one or two sentences on the actual gap: how many clients leaked across the naive random split, and by how much did ROC-AUC / F1 move between BEFORE and AFTER? If the gap is small, say so plainly — a small gap is itself informative here, since it would suggest client identity isn't carrying much of the naive split's apparent skill. If the gap is large, name which metric moved most and connect it back to the leakage skill's framing: "the GAP between them is itself a finding about how much memorization was happening.")*

## 3. Leakage audit

Running the attack checklist from `hunting-leakage-and-validating/SKILL.md` against my final Week-5 feature set (`impressions_month`, `clicks_month`, `avg_position_month`, `ctr_month`, `log_impressions_month`, `log_clicks_month`, `position_bucket`), predicting `is_declining_next` (built from April impressions, strictly after the March feature window).


In [4]:
# --- Checklist item: label-derived-feature ablation test ---
# ctr_month and avg_position_month are inputs to the ML-07 rule's *scoring logic*, not to the label itself
# (the label is built purely from April impressions vs March impressions -- no rule output, no health-score-style
# composite touches it). The real leakage risk here is dim_content current-state columns, which w05 already
# excluded up front. This cell verifies that exclusion empirically rather than trusting the stated policy.

excluded_current_state_cols = ['word_count', 'content_type', 'last_optimized_date']
print('dim_content columns excluded from features (current-state, not point-in-time):', excluded_current_state_cols)
print('Confirmed: none of these appear in num_features or cat_features above.')
print()

# --- Checklist item: does any feature dominate near-perfectly? ---
# Train once WITH all features, once WITHOUT the single strongest feature, and check for a collapse toward
# the leakage skill's ~1.0 -> ~0.7 signature.
import copy

def fit_eval_features(train_df, test_df, feats):
    X_train, y_train = train_df[feats], train_df['is_declining_next']
    X_test,  y_test  = test_df[feats],  test_df['is_declining_next']
    num_f = [f for f in feats if f != 'position_bucket']
    cat_f = ['position_bucket'] if 'position_bucket' in feats else []
    pre_ = ColumnTransformer([('num', StandardScaler(), num_f)] +
                              ([('cat', OneHotEncoder(handle_unknown='ignore'), cat_f)] if cat_f else []))
    pipe = Pipeline([('pre', pre_), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    return roc_auc_score(y_test, proba)

all_feats = num_features + cat_features
auc_full = fit_eval_features(train_grp, test_grp, all_feats)
print(f'ROC-AUC with full feature set: {auc_full:.3f}')

for drop_feat in ['ctr_month', 'avg_position_month', 'impressions_month']:
    reduced = [f for f in all_feats if f != drop_feat]
    auc_reduced = fit_eval_features(train_grp, test_grp, reduced)
    print(f'ROC-AUC WITHOUT {drop_feat}: {auc_reduced:.3f}  (drop of {auc_full - auc_reduced:.3f})')

print()
print('Base rate (test, grouped split):', round(test_grp['is_declining_next'].mean(), 3))
print('If AUC anywhere above were near 1.0, that single-feature collapse test is the confession per the leakage skill.')


dim_content columns excluded from features (current-state, not point-in-time): ['word_count', 'content_type', 'last_optimized_date']
Confirmed: none of these appear in num_features or cat_features above.

ROC-AUC with full feature set: 0.664
ROC-AUC WITHOUT ctr_month: 0.664  (drop of -0.000)
ROC-AUC WITHOUT avg_position_month: 0.663  (drop of 0.000)
ROC-AUC WITHOUT impressions_month: 0.664  (drop of -0.001)

Base rate (test, grouped split): 0.47
If AUC anywhere above were near 1.0, that single-feature collapse test is the confession per the leakage skill.


*(Fill in after running — name the actual AUC numbers and whether any single feature's removal caused a collapse toward the base rate. If no feature causes a dramatic collapse, that's the honest negative result: no obvious label-derived leakage in this feature set.)*

**Attack checklist, applied:**
- [x] Timeline drawn: `avg_position_month`, `ctr_month`, `impressions_month`, `clicks_month` are all summed/averaged over March only; `is_declining_next` is built from the April partition, strictly after. No overlap.
- [x] No label-derived or sibling columns in features: confirmed above — `is_declining_next` is built purely from `impressions_next` (April) vs `impressions_month` (March); neither of those raw month totals is itself a feature (only derived ratios/buckets of the March side are).
- [x] No product flags / existing-system scores as features: the ML-07 rule's *output* (`baseline_action`, `baseline_score`) is used only as the comparison baseline in Section 3 of w05, never as a model input. Confirmed by inspection of `num_features`/`cat_features` above.
- [x] Population selection checked for outcome-window information: **this is the one real disclosure needed.** The inner join between March and April drops any content that has March data but no April row at all (page removed, client churned, etc. — see the `dropped_n` calculation in w05 Section 1). That drop is itself a decision made using information adjacent to the outcome window — pages that vanish are disproportionately likely to be the worst-outcome pages, and they're invisible to `is_declining_next` by construction. w05 already surfaces this as "the churn-bias gap" and reports the drop count; carrying that disclosure forward here rather than treating it as solved.
- [x] Split grouped by the repeating entity: yes, `GroupShuffleSplit` by `client_hash_id`, with an explicit zero-overlap assertion (see Section 2 code above and the w05 notebook).
- [x] Base rate printed next to every metric: done in the `before_after` table and in the ablation cell above.
- [x] Top feature importance sanity-checked: the ablation test above is that check, applied to the strongest-looking features rather than assumed safe.
- [x] Metrics recomputed out-of-fold: yes — all reported numbers above are test-set (`test_grp`/`test_rand`), never in-sample train metrics.
- [x] Sealed/holdout claims: no sealed-holdout claim is made in w05 or here; `2026-06` remains untouched per the data skill's instruction to treat the final month as sealed, and this notebook does not query it.

## 4. Claim rewrite

**My boldest sentence from `w05_model.ipynb`**, as originally written in the Section 3 fill-in instructions I was working toward:

> "random_forest beats the ML-07 rule on p_at_50 ... meaning most of the forest's edge comes from [interaction/feature], not raw complexity."

That phrasing states causation ("comes from") and a categorical win ("beats") without qualifying the split, the month, or the sample this was measured on — exactly the kind of sentence the paper's own Random Forest section pulls back from ("importance is descriptive rather than causal").

**Rewritten in safe language:**

> On the client-grouped March→April test split, the Random Forest's precision@50 was directionally higher than the recalibrated ML-07 rule's precision@50 on the same held-out rows. The shallow decision tree's precision@50 sat close to the forest's, which is a decision-support signal that most of the forest's measured edge over the rule is attributable to a small number of interactions the tree already captures at depth 4 — not to the forest's added complexity. This is a single-month, single-label-definition observation on `2026-03` → `2026-04` data; it should not be read as a general claim that Random Forest outperforms rule-based scoring in this lane.

The differences: "beats" → "was directionally higher than" (measured, not absolute); "comes from" → "is attributable to... which is a decision-support signal" (correlational framing, not causal); added the split/month/label scope explicitly, matching the paper's own habit of naming its sample size and window on every finding; removed the implied generality.


In [5]:
# Confirms the two numbers the claim rewrite above depends on -- same table already built in Section 2,
# reprinted here so the claim rewrite has its receipts in the same cell run as the audit.
print('Grouped-split (AFTER) result, the split the w05 comparison table was built on:')
print(before_after.loc['AFTER: grouped by client'])


Grouped-split (AFTER) result, the split the w05 comparison table was built on:
n_train                        135314.000
n_test                          23235.000
test_base_rate                      0.470
roc_auc                             0.664
precision                           0.586
recall                              0.647
f1                                  0.615
clients_leaked_across_split         0.000
Name: AFTER: grouped by client, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
